# Support Ticket Classification & Prioritization

## Business Context

Customer support teams often spend a lot of time reading tickets before assigning them to the right team. This notebook shows a simple NLP pipeline that:

- classifies a ticket into `Billing`, `Technical Issue`, `Account`, or `General Query`
- predicts whether the ticket should be treated as `High`, `Medium`, or `Low` priority

The goal is to build a beginner-friendly but real-world style ML workflow that can be shown in interviews or internship reviews.

## 1. Imports

In [ ]:
from pathlib import Path

import pandas as pd

from src.data_preprocessing import load_raw_data, load_and_prepare_data, get_train_test_split
from src.model_training import train_and_evaluate


## 2. Data Loading

In [ ]:
project_root = Path.cwd()
raw_df = load_raw_data(project_root / 'data' / 'customer_support_tickets.csv')
raw_df.head()

## 3. Quick EDA

The original dataset uses different ticket types than the final business categories. We map them into the required four categories and also collapse `Critical` priority into `High`.

In [ ]:
print('Shape:', raw_df.shape)
print('\nOriginal Ticket Type distribution:')
display(raw_df['Ticket Type'].value_counts().to_frame('count'))
print('\nOriginal Ticket Priority distribution:')
display(raw_df['Ticket Priority'].value_counts().to_frame('count'))

## 4. Preprocessing

In [ ]:
prepared_df = load_and_prepare_data(project_root / 'data' / 'customer_support_tickets.csv')
prepared_df[['ticket_text', 'clean_ticket_text', 'category_label', 'priority_label']].head()

The preprocessing step includes:

- lowercasing
- punctuation removal
- tokenization
- stopword removal with NLTK
- optional lemmatization when NLTK WordNet resources are available

In [ ]:
print('Mapped category distribution:')
display(prepared_df['category_label'].value_counts().to_frame('count'))
print('\nMapped priority distribution:')
display(prepared_df['priority_label'].value_counts().to_frame('count'))

## 5. Train/Test Split

In [ ]:
train_df, test_df = get_train_test_split(prepared_df)
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

## 6. Model Training

The pipeline compares:

- TF-IDF vs Bag of Words
- Logistic Regression
- Multinomial Naive Bayes
- Random Forest

Two separate models are trained:

- one for category classification
- one for priority prediction

In [ ]:
results = train_and_evaluate()
results['model_comparison'].head(10)

## 7. Evaluation Results

In [ ]:
comparison_df = pd.read_csv(project_root / 'outputs' / 'metrics' / 'model_comparison.csv')
comparison_df.sort_values(['task', 'f1_macro', 'accuracy'], ascending=[True, False, False])

Confusion matrix images and class distribution plots are saved in `outputs/visuals/`.

In [ ]:
predictions_df = pd.read_csv(project_root / 'outputs' / 'predictions' / 'predictions.csv')
predictions_df.head()

## 8. Final Insights

- Ticket subject + description together provide enough signal for basic ticket triage.
- Classical ML with TF-IDF is a strong baseline for support text.
- The project is useful for routing tickets and identifying urgent cases earlier.
- Priority mapping is transparent because it is based on the original dataset plus a rule-based fallback when needed.